In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [15]:
df = pd.read_csv("imdb-films-by-actor-for-10k-actors/actorfilms.csv")
df.columns

FileNotFoundError: [Errno 2] No such file or directory: 'imdb-films-by-actor-for-10k-actors/actorfilms.csv'

In [ ]:
df.isnull().sum()

In [ ]:
!pip install node2vec -q

In [ ]:
import pandas as pd
import networkx as nx
import community
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

class FilmNetworkAnalysis:
    def __init__(self, data):
        """
        Initialize with pandas DataFrame containing:
        Actor, ActorID, Film, Year, Votes, Rating, FilmID
        """
        self.data = data
        self.G = nx.Graph()
        self.construct_network()

    def construct_network(self):
        """Construct the multi-modal network"""
        # Convert ActorID and FilmID to strings to ensure consistent node types
        self.data['ActorID'] = self.data['ActorID'].astype(str)
        self.data['FilmID'] = self.data['FilmID'].astype(str)
        
        # Add actor nodes with attributes
        for _, row in self.data[['ActorID', 'Actor']].drop_duplicates().iterrows():
            self.G.add_node(row['ActorID'], 
                           type='actor',
                           name=row['Actor'])

        # Add film nodes with attributes
        for _, row in self.data[['FilmID', 'Film', 'Year', 'Rating', 'Votes']].drop_duplicates().iterrows():
            self.G.add_node(row['FilmID'],
                           type='film',
                           name=row['Film'],
                           year=row['Year'],
                           rating=row['Rating'],
                           votes=row['Votes'])

        # Add edges between actors and films
        edges = self.data[['ActorID', 'FilmID']].drop_duplicates().values.tolist()
        self.G.add_edges_from(edges)

    def analyze_communities(self):
        """Detect communities using Louvain method"""
        communities = community.best_partition(self.G)
        nx.set_node_attributes(self.G, communities, 'community')
        return communities

    def visualize_network(self, layout_type='spring', seed=42, show_labels=False):
        """
        Create optimized network visualization
        
        Parameters:
        layout_type: str, options: 'spring', 'kamada', 'circular', 'random'
        seed: int, random seed for reproducibility
        show_labels: bool, whether to display node labels
        """
        plt.figure(figsize=(15, 10))
        
        # Choose layout based on network size
        n_nodes = self.G.number_of_nodes()
        
        # Calculate layout positions based on type
        if layout_type == 'spring':
            pos = nx.spring_layout(
                self.G,
                k=1.5/np.sqrt(n_nodes),
                iterations=25,
                seed=seed
            )
        elif layout_type == 'kamada':
            pos = nx.kamada_kawai_layout(self.G)
        elif layout_type == 'circular':
            pos = nx.circular_layout(self.G)
        else:
            pos = nx.random_layout(self.G, seed=seed)
        
        # Pre-compute node lists
        actor_nodes = []
        film_nodes = []
        
        for node, attr in self.G.nodes(data=True):
            if attr['type'] == 'actor':
                actor_nodes.append(node)
            else:
                film_nodes.append(node)
        
        # Draw nodes in batches
        if actor_nodes:
            nx.draw_networkx_nodes(self.G, pos,
                                   nodelist=actor_nodes,
                                   node_color='lightblue',
                                   node_size=100,
                                   alpha=0.9,
                                   label='Actors')
        
        if film_nodes:
            nx.draw_networkx_nodes(self.G, pos,
                                   nodelist=film_nodes,
                                   node_color='lightgreen',
                                   node_size=150,
                                   alpha=0.9,
                                   label='Films')
        
        # Draw edges
        edges = list(self.G.edges())
        if edges:
            nx.draw_networkx_edges(self.G, pos,
                                   edgelist=edges,
                                   alpha=0.5,
                                   width=1)
        
        # Optionally show labels
        if show_labels:
            labels = {node: self.G.nodes[node]['name'] for node in self.G.nodes()}
            nx.draw_networkx_labels(self.G, pos, labels=labels, font_size=10, font_color='black')
        
        plt.title("Film-Actor Network Visualization")
        plt.legend()
        plt.gca().set_axis_off()
        plt.tight_layout()
        
        return plt

    def analyze_rating_votes_correlation(self):
        """Calculate correlation between ratings and votes and plot"""
        correlation, p_value = stats.pearsonr(self.data['Rating'], self.data['Votes'])
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=self.data, x='Rating', y='Votes', alpha=0.6)
        plt.title("Rating vs. Votes Correlation")
        plt.xlabel("Rating")
        plt.ylabel("Votes")
        plt.show()
        
        return correlation, p_value, plt

    def basic_network_metrics(self):
        """Print basic network metrics"""
        num_nodes = self.G.number_of_nodes()
        num_edges = self.G.number_of_edges()
        density = nx.density(self.G)
        
        print(f"Number of nodes: {num_nodes}")
        print(f"Number of edges: {num_edges}")
        print(f"Network density: {density:.4f}")
    
    def analyze_actor_connectivity(self):
        """Analyze actor connectivity and plot degree distribution"""
        actor_nodes = [n for n, attr in self.G.nodes(data=True) if attr['type'] == 'actor']
        actor_degrees = [self.G.degree(node) for node in actor_nodes]
        actor_names = [self.G.nodes[node]['name'] for node in actor_nodes]
        
        # Plot actor degree distribution
        plt.figure(figsize=(10, 6))
        plt.hist(actor_degrees, bins=30)
        plt.xlabel('Number of Films')
        plt.ylabel('Number of Actors')
        plt.title('Actor Degree Distribution')
        plt.show()
        
        # Return top actors by number of films
        top_actors = pd.DataFrame({
            'Actor': actor_names,
            'Films': actor_degrees
        }).sort_values('Films', ascending=False)
        
        return top_actors.head(10)

    def film_statistics_by_year(self):
        """Calculate and return film statistics grouped by year"""
        film_nodes = [(n, attr) for n, attr in self.G.nodes(data=True) if attr['type'] == 'film']
        film_data = pd.DataFrame([
            {'Year': attr['year'], 'Rating': attr['rating'], 'Votes': attr['votes']} 
            for n, attr in film_nodes
        ])
        
        yearly_stats = film_data.groupby('Year').agg({
            'Rating': ['mean', 'count'],
            'Votes': 'mean'
        }).round(2)
        
        return yearly_stats

    def predict_links(self):
        """Predict potential collaborations based on graph structure"""
        preds = nx.jaccard_coefficient(self.G, ebunch=[
            (u, v) for u in self.G.nodes if self.G.nodes[u]['type'] == 'actor'
            for v in self.G.nodes if self.G.nodes[v]['type'] == 'actor' and not self.G.has_edge(u, v)
        ])
        predictions = sorted(preds, key=lambda x: x[2], reverse=True)[:10]
        
        return predictions

    def rating_distribution_by_community(self):
        """Visualize rating distribution by community"""
        film_communities = {
            node: {
                'community': self.G.nodes[node].get('community', -1),  # -1 if community not assigned
                'rating': self.G.nodes[node]['rating']
            }
            for node in self.G.nodes if self.G.nodes[node]['type'] == 'film'
        }
        
        community_ratings = pd.DataFrame.from_dict(film_communities, orient='index')
        
        plt.figure(figsize=(12, 6))
        sns.boxplot(x='community', y='rating', data=community_ratings)
        plt.title('Rating Distribution by Community')
        plt.xlabel('Community ID')
        plt.ylabel('Rating')
        plt.xticks(rotation=90)
        plt.show()

    def calculate_centrality_metrics(self):
        """Calculate and return top actors by betweenness centrality"""
        actor_betweenness = nx.betweenness_centrality(self.G)
        actor_centrality = {
            self.G.nodes[node]['name']: centrality
            for node, centrality in actor_betweenness.items()
            if self.G.nodes[node]['type'] == 'actor'
        }
        
        top_central_actors = pd.DataFrame({
            'Actor': actor_centrality.keys(),
            'Centrality': actor_centrality.values()
        }).sort_values('Centrality', ascending=False)
        
        return top_central_actors.head(10)

    def time_based_analysis(self):
        """Visualize film ratings over time with vote size"""
        film_nodes = [(n, attr) for n, attr in self.G.nodes(data=True) if attr['type'] == 'film']
        film_data = pd.DataFrame([
            {'Year': attr['year'], 'Rating': attr['rating'], 'Votes': attr['votes']} 
            for n, attr in film_nodes
        ])
        
        plt.figure(figsize=(12, 6))
        sns.scatterplot(data=film_data, x='Year', y='Rating', size='Votes', sizes=(20, 200), alpha=0.6)
        plt.title('Film Ratings Over Time (Size = Number of Votes)')
        plt.show()


In [ ]:
# intialize the network

network = FilmNetworkAnalysis(df)

In [ ]:
# Detect communities
communities = network.analyze_communities()
print("Communities detected:", communities)

In [ ]:
# Visualize network
network.visualize_network(layout_type='circular')

In [ ]:
# Analyze rating-votes correlation
correlation, p_value, _ = network.analyze_rating_votes_correlation()
print(f"Correlation: {correlation}, P-value: {p_value}")

In [ ]:
# Get basic network metrics
network.basic_network_metrics()

In [ ]:
# Analyze actor connectivity
top_actors = network.analyze_actor_connectivity()
print("Top 10 Actors by number of films:")
print(top_actors)

In [ ]:
# Get yearly film statistics
yearly_stats = network.film_statistics_by_year()
print("Yearly Film Statistics:")
print(yearly_stats)


In [ ]:
# Predict potential collaborations
predictions = network.predict_links()
print("Predicted Actor Collaborations:")
print(predictions)


In [ ]:
# Rating distribution by community
network.rating_distribution_by_community()

In [ ]:
# Centrality metrics
top_central_actors = network.calculate_centrality_metrics()
print("Top 10 Actors by Betweenness Centrality:")
print(top_central_actors)


In [ ]:
# Time-based analysis
network.time_based_analysis()